In [ ]:
from google.cloud import bigquery
import warnings
import pandas as pd
import re
from datasets import Dataset, load_dataset
from utils.hub_card import push_dataset_card
warnings.filterwarnings(action='ignore', message="Your application has authenticated using end user")

client = bigquery.Client(project="ads-599-final")

In [ ]:
# Medications dispensed from the Pyxis automated cabinet for ED-only patients
# (disposition = 'HOME', hadm_id IS NULL — not admitted to the hospital).
# Pyxis dispenses map to the administer_medication action — the only medication
# action available during an ED stay (no drips, rate changes, or stops in ED).
# Note: Dispensing from Pyxis does not guarantee administration;
# nurses may return medications. This is the best available signal for ED-only patients.

query_ed_only_meds = """
SELECT
  e.subject_id,
  e.stay_id        AS ed_stay_id,
  e.hadm_id,
  e.disposition,
  p.charttime,
  p.med_rn,
  p.name           AS medication,
  TRUE             AS in_er
FROM `physionet-data.mimiciv_ed.edstays` e
JOIN `physionet-data.mimiciv_ed.pyxis` p
  ON e.stay_id = p.stay_id
WHERE e.disposition = 'HOME'
  AND e.hadm_id IS NULL
  AND p.charttime BETWEEN e.intime AND e.outtime
ORDER BY e.subject_id, e.stay_id, p.charttime
"""

print("Running ED-only meds query...")
df_ed_only_meds = client.query(query_ed_only_meds).to_dataframe()
print(f"Shape: {df_ed_only_meds.shape}")
print(f"Unique patients: {df_ed_only_meds['subject_id'].nunique():,}")
print(f"Unique ED stays: {df_ed_only_meds['ed_stay_id'].nunique():,}")
print(f"\nTop medications:\n{df_ed_only_meds['medication'].value_counts().head(10)}")
df_ed_only_meds.head()

In [ ]:
DESCRIPTION = (
    "Medications administered during the ED stay for ED-only patients (not admitted to hospital). "
    "Derived from ed.pyxis (Pyxis dispense records). "
    "Mapped to administer_medication action — the only medication action available during an ED stay."
)

ds = Dataset.from_pandas(df_ed_only_meds, split='ed_only')
ds.push_to_hub("ADS599-Capstone/raw_data", config_name="ed_only_meds", data_dir="stay_medications")
push_dataset_card("ADS599-Capstone/raw_data", config_name="ed_only_meds", split="ed_only", data_dir="stay_medications", description=DESCRIPTION)
print("Pushed ed_only_meds to HuggingFace Hub.")

In [ ]:
# 1. Drug class mapping via REGEX on medication name
# 
# ed_only_meds has no etcdescription column. Pyxis dispense records store only
# the raw medication name string. Drug classes are assigned via 22 regex patterns
# on the name, using the same 22-class vocabulary as medrecon.
#
# Why regex on name (not etcdescription):
#   The Pyxis table is a dispense log, not a pharmacy order system. It was never
#   linked to the ETC taxonomy used in medrecon. Regex on the name string is the
#   only available classification path.
#
# Why the same 22 classes:
#   recon_* and edmeds_* features must occupy the same feature space for the RL
#   state vector to be coherent. A patient's pre-visit beta blocker flag and their
#   in-ED beta blocker dispense flag should be directly comparable.
#   Coverage: 100% of records matched (no 'Unknown' residual after mapping).

NAME_TO_CLASS = {
    r'morphine|hydromorphone|dilaudid|fentanyl|oxycodone|tramadol|ketorolac|toradol': 'Analgesic - Opioid/NSAID',
    r'acetaminophen|tylenol': 'Analgesic - Acetaminophen',
    r'ibuprofen|naproxen': 'Analgesic - NSAID',
    r'ondansetron|zofran|promethazine|phenergan|metoclopramide|reglan|prochlorperazine': 'Antiemetic',
    r'vancomycin|ceftriaxone|cefazolin|piperacillin|zosyn|azithromycin|ciprofloxacin|metronidazole|flagyl|ampicillin|meropenem|levofloxacin': 'Antibiotic',
    r'lorazepam|ativan|diazepam|valium|midazolam|versed|alprazolam': 'Benzodiazepine - Sedative/Anxiolytic',
    r'heparin|enoxaparin|lovenox|warfarin|coumadin|apixaban|rivaroxaban': 'Anticoagulant',
    r'metoprolol|lopressor|labetalol|atenolol|carvedilol': 'Beta Blocker',
    r'lisinopril|enalapril|captopril|ramipril': 'ACE Inhibitor',
    r'amlodipine|diltiazem|verapamil|nifedipine': 'Calcium Channel Blocker',
    r'nitroglycerin|nitro': 'Nitrate',
    r'amiodarone|adenosine|digoxin': 'Antiarrhythmic',
    r'furosemide|lasix|bumetanide|torsemide|hydrochlorothiazide': 'Diuretic',
    r'methylprednisolone|prednisone|dexamethasone|hydrocortisone': 'Corticosteroid',
    r'pantoprazole|omeprazole|famotidine|pepcid|ranitidine': 'GI - Acid Suppression',
    r'albuterol|ipratropium|atrovent|levalbuterol': 'Bronchodilator',
    r'insulin|dextrose|glucagon': 'Insulin/Glucose',
    r'haloperidol|haldol|olanzapine|quetiapine|risperidone': 'Antipsychotic',
    r'levetiracetam|keppra|phenytoin|dilantin|valproate|depakote|lacosamide': 'Anticonvulsant',
    r'normal saline|sodium chloride 0\.9|lactated|ringer|d5w|dextrose 5': 'IV Fluid',
    r'aspirin|clopidogrel|plavix|ticagrelor': 'Antiplatelet',
}

def map_name_to_class(name):
    if pd.isna(name):
        return 'Other'
    name_lower = str(name).lower()
    for pattern, drug_class in NAME_TO_CLASS.items():
        if re.search(pattern, name_lower):
            return drug_class
    return 'Other'

df_ed_only_meds['drug_class'] = df_ed_only_meds['medication'].apply(map_name_to_class)

n_mapped = (df_ed_only_meds['drug_class'] != 'Other').sum()
print(f'Class assigned: {n_mapped:,} ({n_mapped/len(df_ed_only_meds)*100:.1f}%)')
print(f'Other (unmatched): {(df_ed_only_meds["drug_class"]=="Other").sum():,}')
print('\nClass distribution:')
print(df_ed_only_meds['drug_class'].value_counts())

In [ ]:
# 2. Compute MINUTES_INTO_STAY (timestep for RL trajectory)
# 
# Each dispense event is placed on the RL timeline by computing how many minutes
# after ED arrival it occurred. This is the timestep at which action_idx fires.
#
# Why minutes not clock time:
#   The RL agent operates within an episode starting at ED arrival. Absolute clock
#   time is meaningless across patients and time-since-arrival is the shared scale.
#
# Window filter (0 to 1440 min = 24 hours):
#   Records outside this window are data quality artifacts (timestamp errors).
#   The 24-hour ceiling covers the longest plausible ED-only stay.

# Load cohort to get ed_intime for each visit
cohort = load_dataset('ADS599-Capstone/raw_data', 'cohort_base', split = 'base',
                      verification_mode = 'no_checks').to_pandas()
cohort['ed_intime'] = pd.to_datetime(cohort['ed_intime'])

df_ed_only_meds['charttime'] = pd.to_datetime(df_ed_only_meds['charttime'])
df_timed = df_ed_only_meds.merge(
    cohort[['ed_stay_id', 'ed_intime']],
    left_on = 'ed_stay_id', right_on = 'ed_stay_id', how = 'left'
)
df_timed['minutes_into_stay'] = (
    (df_timed['charttime'] - df_timed['ed_intime']).dt.total_seconds() / 60
)
df_timed = df_timed[
    (df_timed['minutes_into_stay'] >= 0) &
    (df_timed['minutes_into_stay'] <= 1440)
].copy()
print(f'Records after window filter: {len(df_timed):,}')
print(f'Minutes into stay — mean: {df_timed["minutes_into_stay"].mean():.1f}, '
      f'median: {df_timed["minutes_into_stay"].median():.1f}, '
      f'max: {df_timed["minutes_into_stay"].max():.0f}')

In [ ]:
# 3. Build action index and save action file
#
# Each drug class is assigned a stable integer action_idx (0-21).
# This is the discrete action the RL agent selects at each timestep.
#
# Why integer not one-hot:
#   CQL, IQL, and Behavior Cloning all expect a scalar action from a discrete
#   action space. The action_vocab.csv file maps idx back to drug_class for
#   interpretability.
#
# Schema: ed_stay_id | subject_id | charttime | minutes_into_stay |
#         drug_class | action_idx | source
# This schema is shared with the meds_admitted action file so both can be
# concatenated directly for full trajectory construction.

class_vocab = sorted(df_timed['drug_class'].dropna().unique().tolist())
class_to_idx = {c: i for i, c in enumerate(class_vocab)}

df_timed['action_idx'] = df_timed['drug_class'].map(class_to_idx).astype(int)
df_timed['source'] = 'ed_only_meds'

action_out = df_timed[[
    'ed_stay_id', 'subject_id', 'charttime', 'minutes_into_stay',
    'drug_class', 'action_idx', 'source'
]].copy()

action_out.to_csv('features_ed_meds_timestep_actions.csv', index = False)
pd.DataFrame({'action_idx': list(class_to_idx.values()),
              'drug_class':  list(class_to_idx.keys())
              }).to_csv('action_vocab.csv', index = False)
print(f'Saved features_ed_meds_timestep_actions.csv  shape: {action_out.shape}')
print(f'Action space: {len(class_vocab)} drug classes')
print(f'Saved action_vocab.csv')

In [ ]:
# 4. Build cumulative binary feature matrix (static snapshot per visit)
# 
# For the RL state vector we also need a static summary of which drug classes
# were dispensed at any point during the visit. Used as cumulative context
# features when the full timestep trajectory is not needed.
#
# Why cumulative binary (not count):
#   Same reasoning as medrecon,  the presence/absence is the clinically meaningful
#   signal. The count of dispenses is captured separately as edmeds_n_dispenses.

def safe_col(s):
    return (str(s).lower()
            .replace(' ', '_').replace('-', '_').replace('/', '_')
            .replace('(', '').replace(')', '').replace(',', '')
            .replace('&', 'and')[:60])

pairs = (df_timed[['ed_stay_id', 'drug_class']]
         .drop_duplicates()
         .assign(flag = 1))
ed_pivot = (pairs
            .set_index(['ed_stay_id', 'drug_class'])['flag']
            .unstack(fill_value = 0)
            .astype('uint8'))
ed_pivot.columns = [f'edmeds_{safe_col(c)}' for c in ed_pivot.columns]
ed_pivot = ed_pivot.reset_index()

ed_n = (df_timed.groupby('ed_stay_id')
        .agg(edmeds_n_dispenses=('drug_class', 'count'),
             edmeds_n_drug_classes=('drug_class', 'nunique'))
        .reset_index())

subject_map = df_timed[['ed_stay_id', 'subject_id']].drop_duplicates('ed_stay_id')
ed_features = subject_map.merge(ed_pivot, on = 'ed_stay_id', how = 'right').merge(ed_n, on = 'ed_stay_id', how = 'left')

ed_features.to_csv('features_ed_meds_cumulative.csv', index = False)
print(f'Saved features_ed_meds_cumulative.csv  shape: {ed_features.shape}')